# Energy Consumption Time Series Forecasting


## 1. Problem Statement and Objective

This project forecasts short-term household electricity consumption using historical power usage data. The objective is to compare traditional statistical forecasting, additive time-series forecasting, and machine learning forecasting models.

## 2. Dataset Description

The dataset is the Individual Household Electric Power Consumption Dataset. It contains minute-level measurements for household electricity usage, voltage, intensity, and sub-metering values.

Target variable: `Global_active_power`.

## 3. Import Libraries

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from config import FORECASTS_DIR, METRICS_DIR, TARGET_COLUMN
from src.data_loader import load_power_consumption_data
from src.evaluation import build_comparison_table
from src.feature_engineering import create_features
from src.modeling import run_all_models
from src.preprocessing import chronological_train_test_split, clean_missing_values, limit_recent_data, resample_power_data

sns.set_theme(style='darkgrid', palette='viridis')
plt.rcParams['figure.figsize'] = (14, 6)


## 4. Load Dataset

In [ ]:
raw_df = load_power_consumption_data()
raw_df.head()


In [ ]:
print(f'Records: {len(raw_df):,}')
print(f'Date range: {raw_df.index.min()} to {raw_df.index.max()}')
raw_df.describe().T


## 5. Data Cleaning and Preprocessing

In [ ]:
missing_before = raw_df.isna().sum().sort_values(ascending=False)
missing_before


In [ ]:
clean_df = clean_missing_values(raw_df)
hourly_df = resample_power_data(clean_df, frequency='h')
daily_df = resample_power_data(clean_df, frequency='D')

hourly_df.head()


## 6. Exploratory Data Analysis

In [ ]:
missing_before.plot(kind='bar', color='#10B981')
plt.title('Missing Values by Column')
plt.ylabel('Missing Count')
plt.tight_layout()
plt.show()


In [ ]:
hourly_df[TARGET_COLUMN].plot(color='#10B981')
plt.title('Hourly Energy Consumption Trend')
plt.ylabel('Global Active Power')
plt.tight_layout()
plt.show()


In [ ]:
daily_df[TARGET_COLUMN].plot(color='#047857')
plt.title('Daily Average Consumption Trend')
plt.ylabel('Global Active Power')
plt.tight_layout()
plt.show()


In [ ]:
monthly_avg = clean_df[[TARGET_COLUMN]].resample('M').mean()
monthly_avg[TARGET_COLUMN].plot(marker='o', color='#34D399')
plt.title('Monthly Average Consumption Trend')
plt.ylabel('Global Active Power')
plt.tight_layout()
plt.show()


In [ ]:
eda_df = hourly_df.copy()
eda_df['hour'] = eda_df.index.hour
eda_df['day_of_week'] = eda_df.index.dayofweek
eda_df['is_weekend'] = eda_df['day_of_week'].isin([5, 6]).map({True: 'Weekend', False: 'Weekday'})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(data=eda_df, x='hour', y=TARGET_COLUMN, ax=axes[0], color='#10B981')
axes[0].set_title('Average Consumption by Hour of Day')
sns.barplot(data=eda_df, x='day_of_week', y=TARGET_COLUMN, ax=axes[1], color='#047857')
axes[1].set_title('Average Consumption by Day of Week')
sns.barplot(data=eda_df, x='is_weekend', y=TARGET_COLUMN, ax=axes[2], palette=['#10B981', '#34D399'])
axes[2].set_title('Weekend vs Weekday Consumption')
plt.tight_layout()
plt.show()


## 7. Feature Engineering

In [ ]:
# Use a recent sample for practical notebook runtime. Switch sample_days to None for full data.
model_df = limit_recent_data(hourly_df, days=180)
features_df = create_features(model_df)
features_df.head()


## 8. Train-Test Split

In [ ]:
train_df, test_df = chronological_train_test_split(model_df, test_size=0.2)
train_features = features_df.loc[features_df.index.isin(train_df.index)]
test_features = features_df.loc[features_df.index.isin(test_df.index)]

print(f'Train rows: {len(train_df):,}')
print(f'Test rows: {len(test_df):,}')


## 9. Model 1: ARIMA

ARIMA is a statistical baseline model. The project uses a stable default order of `(2, 1, 2)` and trains on a recent sample when needed.

## 10. Model 2: Prophet

Prophet models trend and seasonality. If Prophet is unavailable, it is skipped gracefully.

## 11. Model 3: XGBoost

XGBoost uses time features, lag features, and rolling statistics to learn short-term usage patterns.

In [ ]:
forecasts, messages = run_all_models(
    train=train_df,
    test=test_df,
    train_features=train_features,
    test_features=test_features,
    include_prophet=True,
)

for message in messages:
    print(message)

list(forecasts.keys())


## 12. Model Evaluation

In [ ]:
metrics_df = build_comparison_table(forecasts, test_df[TARGET_COLUMN])
metrics_df


In [ ]:
METRICS_DIR.mkdir(parents=True, exist_ok=True)
FORECASTS_DIR.mkdir(parents=True, exist_ok=True)

forecast_results = pd.DataFrame({'Actual': test_df[TARGET_COLUMN]})
for model_name, prediction in forecasts.items():
    forecast_results[model_name] = prediction

metrics_df.to_csv(METRICS_DIR / 'model_comparison.csv', index=False)
forecast_results.to_csv(FORECASTS_DIR / 'forecast_results.csv')

forecast_results.head()


## 13. Actual vs Forecasted Visualization

In [ ]:
for model_name in forecasts:
    plt.figure(figsize=(14, 5))
    plt.plot(forecast_results.index, forecast_results['Actual'], label='Actual', color='#111827')
    plt.plot(forecast_results.index, forecast_results[model_name], label=model_name, color='#10B981')
    plt.title(f'Actual vs Forecasted Energy Usage: {model_name}')
    plt.ylabel('Global Active Power')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
metrics_long = metrics_df.melt(id_vars='Model', value_vars=['MAE', 'RMSE', 'MAPE'], var_name='Metric', value_name='Value')
sns.barplot(data=metrics_long, x='Model', y='Value', hue='Metric')
plt.title('Model Comparison by MAE, RMSE, and MAPE')
plt.tight_layout()
plt.show()


## 14. Final Conclusion and Insights

Hourly and daily patterns are important for household energy forecasting. XGBoost usually performs strongly because it uses lag and rolling features. ARIMA is useful as a statistical baseline. Prophet is useful for seasonality and trend when the package is available and the data length is sufficient. Time-based feature engineering improves forecasting quality by exposing recurring usage behavior to the model.